In [ ]:
# Data Manipulation Imports
import numpy as np
import pandas as pd
import xarray as xr
import scipy

# Plotting Imports
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns

from datetime import datetime, date, time, timedelta
import matplotlib.dates as mdates

In [ ]:
# Path to Crocolake
parquet_dir = '/home/jovyan/shared/go-bgc-2026/data/CrocoLake/BGC_CROCOLAKE/'

# path to heatwave csv
csv_path = '../../MHW_list.csv'

dfhw = pd.read_csv(csv_path)
dfhw.set_index('name',inplace=True)
date_cols = ['date_start','date_end']
dfhw[date_cols] = dfhw[date_cols].apply(pd.to_datetime)

# Choose a HW
hw = "NA 2023"

# Boundaries
lat0 = dfhw.loc[hw,'lat0'] 
lat1 = dfhw.loc[hw,'lat1'] 
lon0 = dfhw.loc[hw,'lon0'] 
lon1 = dfhw.loc[hw,'lon1'] 

date0 = dfhw.loc[hw,'date_start']
date1 = dfhw.loc[hw,'date_end']

In [ ]:
%%time

# Parameters
columns = ('PRES','TEMP','PSAL','DOXY','CHLA','BBP700','LATITUDE','LONGITUDE','JULD','CYCLE_NUMBER','DB_NAME','PLATFORM_NUMBER')


# without DB_NAME you also would get any Spray glider or shipboard GLODAP data too
filters = [
    ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
    ("LONGITUDE",">",lon0), ("LONGITUDE","<",lon1),
    ("JULD",">",date0), ("JULD","<",date1),
    ("DB_NAME","=","ARGO") 
]

df = pd.read_parquet(parquet_dir,columns=columns,filters=filters)
df = df.dropna(subset=['DOXY','CHLA','BBP700','PSAL','TEMP'])
df

In [ ]:
df = df[df.PRES <= 300]
df = df[df.CHLA >= 0]

In [ ]:
var_names = ['PRES','TEMP','PSAL','DOXY','CHLA','BBP700','LATITUDE','LONGITUDE','JULD']
PRES, TEMP, PSAL, DOXY, CHLA, BBP700, LATITUDE, LONGITUDE, JULD = var_names

In [ ]:
data_da = df.set_index(JULD).resample('W').mean(numeric_only=True).dropna(how='all').reset_index()
peaks, _ = scipy.signal.find_peaks(data_da[CHLA], distance = 300/7)
full_bloom_slices = []
bloom_slices = []
for peak in peaks:
    bloom_slice = data_da[data_da[JULD].between(data_da[JULD].iloc[peak] - pd.Timedelta(weeks=4*6), data_da[JULD].iloc[peak] + pd.Timedelta(weeks=4*6))]
    bloom_slices.append(bloom_slice)
    full_bloom_slices.append(df[df[JULD].between(data_da[JULD].iloc[peak] - pd.Timedelta(weeks=4*6), data_da[JULD].iloc[peak] + pd.Timedelta(weeks=4*6))])
    

In [ ]:
var_names = ['PRES','TEMP','PSAL','DOXY','CHLA','BBP700','LATITUDE','LONGITUDE','JULD']

def round_to_year(dt):
    # If month is 7 or higher, round up; otherwise, round down
    new_year = dt.year + 1 if dt.month >= 7 else dt.year
    return datetime(new_year, 1, 1)
def find_bloom_phenological_indices(bloom_slice, var_names):
    PRES, TEMP, PSAL, DOXY, CHLA, BBP700, LATITUDE, LONGITUDE, JULD = var_names
    bloom_slice["RATE_OF_CHANGE"] = bloom_slice[CHLA].diff() / bloom_slice[JULD].diff().dt.days
    bloom_max_idx = bloom_slice[CHLA].idxmax()
    bloom_max_value = bloom_slice[CHLA].max()
    pre_slice = bloom_slice.loc[:bloom_max_idx]
    post_slice = bloom_slice.loc[bloom_max_idx:]
    bloom_min_idx = pre_slice[CHLA].idxmin()
    bloom_min_value = pre_slice[CHLA].min()
    bloom_range = bloom_slice[CHLA].max() - bloom_slice[CHLA].min()
    # Bloom Initiations
    bloom_initiation_TS_threshold = bloom_min_value + (0.05 * bloom_range)
    mini_slice = bloom_slice.loc[bloom_min_idx:bloom_max_idx]
    try:
        bloom_initiation_TS = mini_slice.where(mini_slice[CHLA] > bloom_initiation_TS_threshold).dropna(how="all").iloc[0]
    except IndexError:
        bloom_initiation_TS = None
    try:
        bloom_termination_TS = post_slice.where(post_slice[CHLA] < bloom_initiation_TS_threshold).dropna(how="all").iloc[0]
    except IndexError:
        bloom_termination_TS = None
    # mins = scipy.signal.argrelextrema(bloom_slice["CHLA_ADJUSTED_BGCArgoPlus"].values, np.less, order=2)
    mini_slice = bloom_slice.loc[bloom_min_idx:]
    mini_slice_filtered = mini_slice[mini_slice[CHLA] < 3*np.median(bloom_slice[CHLA])]
    chla_cumsum = np.cumsum(mini_slice_filtered[CHLA])[-1]
    try:
        bloom_initiation_CS = mini_slice.where(mini_slice[CHLA] > (chla_cumsum * 0.1)).dropna(how="all").iloc[0]
    except IndexError:
        bloom_initiation_CS = None
    try:
        bloom_termination_CS = post_slice.where(post_slice[CHLA] < (chla_cumsum * 0.1))[1:].dropna(how="all").iloc[0]
    except IndexError:
        bloom_termination_CS = None
    rate_of_change_threshold = 0.15 * np.nanmedian(np.abs(bloom_slice["RATE_OF_CHANGE"]))
    try:
        bloom_initiation_RC = pre_slice.where(pre_slice["RATE_OF_CHANGE"] > rate_of_change_threshold).dropna(how="all").iloc[0]
    except IndexError:
        bloom_initiation_RC = None
    try:
        bloom_termination_RC = post_slice.where(post_slice["RATE_OF_CHANGE"] > -rate_of_change_threshold)[1:].dropna(how="all").iloc[0]
    except IndexError:
        bloom_termination_RC = None
    peaks, _ = scipy.signal.find_peaks(bloom_slice[CHLA], height=bloom_max_value * 0.75, distance = 3)

    phenology_dict = {
        # "REGION": region,
        "YEAR": round_to_year(bloom_slice[JULD].loc[bloom_max_idx]).year,
        # "WMOS": bloom_slice["WMO_ID"].unique(),
        "BLOOM_MAXIMUM": bloom_max_value,
        "BLOOM_PEAKS_VALUES": bloom_slice[CHLA].iloc[peaks].values,
        "BLOOM_PEAKS_JULD": bloom_slice[JULD].iloc[peaks].values,
        "BLOOM_SLICE": [bloom_slice],
        "BLOOM_INITIATION_TS": bloom_initiation_TS[CHLA] if bloom_initiation_TS is not None else None,
        "BLOOM_TERMINATION_TS": bloom_termination_TS[CHLA] if bloom_termination_TS is not None else None,
        "BLOOM_DURATION_TS": (bloom_termination_TS[JULD] - bloom_initiation_TS["JULD"]).days if bloom_initiation_TS is not None and bloom_termination_TS is not None else None,
        "BLOOM_INITIATION_TS_DATE": bloom_initiation_TS[JULD] if bloom_initiation_TS is not None else None,
        "BLOOM_TERMINATION_TS_DATE": bloom_termination_TS[JULD] if bloom_termination_TS is not None else None,
        "BLOOM_INITIATION_CS": bloom_initiation_CS[CHLA] if bloom_initiation_CS is not None else None,
        "BLOOM_TERMINATION_CS": bloom_termination_CS[CHLA] if bloom_termination_CS is not None else None,
        "BLOOM_DURATION_CS": (bloom_termination_CS[JULD] - bloom_initiation_CS["JULD"]).days if bloom_initiation_CS is not None and bloom_termination_CS is not None else None,
        "BLOOM_INITIATION_CS_DATE": bloom_initiation_CS[JULD] if bloom_initiation_CS is not None else None,
        "BLOOM_TERMINATION_CS_DATE": bloom_termination_CS[JULD] if bloom_termination_CS is not None else None,
        "BLOOM_INITIATION_RC": bloom_initiation_RC[CHLA] if bloom_initiation_RC is not None else None,
        "BLOOM_TERMINATION_RC": bloom_termination_RC[CHLA] if bloom_termination_RC is not None else None,
        "BLOOM_DURATION_RC": (bloom_termination_RC[JULD] - bloom_initiation_RC["JULD"]).days if bloom_initiation_RC is not None and bloom_termination_RC is not None else None,
        "BLOOM_INITIATION_RC_DATE": bloom_initiation_RC[JULD] if bloom_initiation_RC is not None else None,
        "BLOOM_TERMINATION_RC_DATE": bloom_termination_RC[JULD] if bloom_termination_RC is not None else None,
    }
    # index_type = 1:
    # index_type = 2:
    # index_type = 3:
    
    return phenology_dict

In [ ]:
i = 0
for bloom_slice in bloom_slices:
    phenology_dict = find_bloom_phenological_indices(bloom_slices[0], var_names)
    p_df = pd.DataFrame([phenology_dict]).reset_index(drop=True)
    if i == 0:
        phenology_df = pd.DataFrame([phenology_dict]).reset_index(drop=True)
    else:
        phenology_df = pd.concat([phenology_df, p_df])
    i += 1

In [ ]:
phenology_df.BLOOM_SLICE.values[0]

In [ ]:
for pdf in p

In [ ]:
i = 0
for bloom_slice in bloom_slices:
    full_slice = full_bloom_slices[i].dropna(subset=[PRES])
    full_slice = full_slice.dropna(subset=[CHLA])
    x = mdates.date2num(full_slice["JULD"].to_numpy())
    y = full_slice[PRES].values.to_numpy()
    z = full_slice[CHLA].values.to_numpy()
    
    xi = np.linspace(x.min(), x.max(), 300)
    yi = np.linspace(y.min(), y.max(), 300)
    X, Y = np.meshgrid(xi, yi)
    Z = scipy.interpolate.griddata((x, y), z, (X, Y), method='cubic')
    fig, (ax1, ax2) = plt.subplots(2, figsize=(10, 15))
    
    ax1.plot(bloom_slice[JULD], bloom_slice[CHLA])
    
    for peak in peaks:
        if bloom_slice[CHLA].iloc[peak] == bloom_max_value:
            ax1.scatter(bloom_slice[JULD].iloc[peak], bloom_slice[CHLA].iloc[peak], color='red', marker='*', label='Main Peak')
        else:
            ax1.scatter(bloom_slice[JULD].iloc[peak], bloom_slice[CHLA].iloc[peak], color='red', marker='o')
    # for valley in valleys:
    try:
        ax1.scatter(bloom_initiation_TS[JULD], bloom_initiation_TS[CHLA], color='green', marker='o', label='Bloom Initiation (TS)')
    except TypeError:
        pass
    try:
        ax1.scatter(bloom_termination_TS[JULD], bloom_termination_TS[CHLA], color='orange', marker='o', label='Bloom Termination (TS)')
    except TypeError:
        pass
    try:
        ax1.scatter(bloom_initiation_CS[JULD], bloom_initiation_CS[CHLA], color='green', marker='x', label='Bloom Initiation (CS)')
    except TypeError:
        pass
    try:
        ax1.scatter(bloom_termination_CS[JULD], bloom_termination_CS[CHLA], color='orange', marker='x', label='Bloom Termination (CS)')
    except TypeError:
        pass
    try:
        ax1.scatter(bloom_initiation_RC[JULD], bloom_initiation_RC[CHLA], color='green', marker='s', label='Bloom Initiation (RC)')
    except TypeError:
        pass
    try:
        ax1.scatter(bloom_termination_RC[JULD], bloom_termination_RC[CHLA], color='orange', marker='s', label='Bloom Termination (RC)')
    except TypeError:
        pass
    ax1.axvline(bloom_slice["JULD"].loc[bloom_min], color='blue', linestyle='--', label='Pre-Bloom Minimum')
    ax1.axhline(0, color='gray', linestyle='--', label='Zero Line')
    ax1.set_xlabel("Date")
    ax1.set_xlim(full_slice["JULD"].min(), full_slice["JULD"].max())
    ax1.set_ylabel("Chlorophyll-a (mg/m^3)")
    ax1.legend()
    # ax1.set_title(f"Region {region} {round_to_year(bloom_slice['JULD'].loc[bloom_max]).year} - Bloom Phenology")
    # plt.tight_layout()
    # plt.savefig(f"region_{region}_{round_to_year(bloom_slice['JULD'].loc[bloom_max]).year}.png")
    # plt.show()
    
    # pc = ax2.pcolormesh(X, Y, Z, shading='auto', cmap='viridis', vmin=0)
    pc = ax2.scatter(full_bloom_slices[i][JULD], full_bloom_slices[i][PRES], c=full_bloom_slices[i]["CHLA_ADJUSTED_BGCArgoPlus"], s=5)
    # 3. Format the x-axis
    # hfmt = mdates.DateFormatter('%Y-%m-%d')
    # ax2.xaxis.set_major_formatter(hfmt)
    
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Pressure (dbar)")
    # ax2.set_title(f"Region {region} {round_to_year(bloom_slice['JULD'].loc[bloom_max]).year} - Bloom Phenology Depth Profile")
    fig.colorbar(pc, label="Chlorophyll-a (mg/m^3)", orientation="horizontal")
    
    ax2.set_ylim(0, 400)
    ax2.invert_yaxis()
    plt.tight_layout()
    plt.show()
    i += 1


In [ ]:
full_slice["JULD"].value

In [ ]:
full_slice[PRES].values